In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

## LER OS RETORNOS, SCORE e SELIC

In [3]:
# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ativos_{ano_usado}.csv'
    caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv(caminho).set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    print(ano,'-',len(dfs[ano_usado].columns))
    

2025-12-31 - 78
2024-12-31 - 78
2023-12-31 - 78
2022-12-31 - 77
2021-12-31 - 77
2020-12-31 - 73
2019-12-31 - 73
2018-12-31 - 70
2017-12-31 - 70
2016-12-31 - 67
2015-12-31 - 62


In [4]:
## SELIC para Sigma e excesso
selic_d = pd.read_csv('selic/selic_diario.csv').set_index('date')

## EXCESSO DOS ANOS e SIGMA (MAtriz de covariancia)

##### EXCESSO PARA TODOS

In [5]:
dict_excesso = {}
for an in anos:
    ano = an.split("-")[0]
    try:
        print(f"=============== \n EXCESSO {ano}\n ============")
        print("Atualização, Ano: ",ano)
        df = dfs[ano]
        # df_f = pd.DataFrame(eval(df))
        df_f = df.copy()
        slc = selic_d[selic_d.index.isin(df_f.index)]
        slc['valor_diario'] = slc['valor_diario']/100

        print(f"Tamanho DF de {ano}: ", len(df_f))
        print(f"Tamanho Selic: ", len(slc['valor_diario']))
        ano = int(ano)
        dict_excesso[ano] = df_f.sub(slc['valor_diario'],axis=0)
        print("tamanho final do Excesso: ",len(dict_excesso[ano]))
    except Exception as e:
        print(e)
        print("ERror")

 EXCESSO 2025
Atualização, Ano:  2025
Tamanho DF de 2025:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 EXCESSO 2024
Atualização, Ano:  2024
Tamanho DF de 2024:  122
Tamanho Selic:  122
tamanho final do Excesso:  122
 EXCESSO 2023
Atualização, Ano:  2023
Tamanho DF de 2023:  121
Tamanho Selic:  121
tamanho final do Excesso:  121
 EXCESSO 2022
Atualização, Ano:  2022
Tamanho DF de 2022:  124
Tamanho Selic:  124
tamanho final do Excesso:  124
 EXCESSO 2021
Atualização, Ano:  2021
Tamanho DF de 2021:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 EXCESSO 2020
Atualização, Ano:  2020
Tamanho DF de 2020:  120
Tamanho Selic:  120
tamanho final do Excesso:  120
 EXCESSO 2019
Atualização, Ano:  2019
Tamanho DF de 2019:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 EXCESSO 2018
Atualização, Ano:  2018
Tamanho DF de 2018:  119
Tamanho Selic:  119
tamanho final do Excesso:  119
 EXCESSO 2017
Atualização, Ano:  2017
Tamanho DF de 2017:  119
Tamanho Selic:  119
taman

## Hiperparâmetros

In [6]:
vb_cardinalidade_max = 10
vb_cardinalidade_min = 10
vb_peso_maximo = 0.20
vb_peso_minimo = 0.02
vb_theta = 0.5

### Fazendo otimização ano a ano e comparando com o proximo ano

In [7]:
anos = ['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']
print(anos)

['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']


In [14]:
y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])

print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)

for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    
        
        #SCORE MF------------

    score_mf_final = pd.read_csv(f'score_mf/{ano}/mf_{ano}.csv').set_index('ano')
    score_usado_mf = score_mf_final.dropna(axis=1).copy()
    # print(score_usado)
    print("Score Atualizado")

        #SCORE PIO------------
    
    score_pio_final = pd.read_csv(f'score_piotroski//{ano}/piotroski_{ano}.csv').set_index('ano')
    score_usado_pio = score_pio_final.dropna(axis=1).copy()
    # print(score_usado)
    print("Score Atualizado")

    COLUNAS_SCORES = score_usado_pio.columns.intersection(score_usado_mf.columns)
    score_usado_pio = score_usado_pio[COLUNAS_SCORES]
    score_usado_mf = score_usado_mf[COLUNAS_SCORES]
    ativos_finais = COLUNAS_SCORES


    ano_um = int(ano)
    # df_usado = f'df_ativos_{ano_um}'
    # retorno_usado = pd.DataFrame(eval(df_usado))
    df_usado = dfs[str(ano_um)][ativos_finais]
    retorno_usado = df_usado.copy()
    # retorno_usado = retorno_usado[lista_ativos_finais]
    print("Retornos atualizados")

    # sigma_usado = retorno_usado.cov()
    # print("Sigmas Atualizados")
    print("-----")
    

    # if df_usado == 'df_ativos_2015':
    #     continue
    # else:
    print(f"{an} TAMANHOS: Retorno {len(retorno_usado.columns)}, Score PIO {len(score_usado_pio.columns)}, Score MF {len(score_usado_mf.columns)}")
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",int(ano)+1)

    # # print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    # print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    # # print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    # print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns.tolist())
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns.tolist())-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    # model.theta = pyo.Param(initialize=vb_theta)
    model.score_p = pyo.Param(model.ativos, initialize=lambda model,a: score_usado_pio.iloc[0,a])
    model.score_m = pyo.Param(model.ativos, initialize=lambda model,a: score_usado_mf.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos, bounds=(0,1))
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    # model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    # model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    # model.s = pyo.Param(initialize = 2, mutable=True)
    # model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    for i in range(3):
        setattr(model,f'obj{i}_mais',pyo.Var(bounds=(0,0.1),domain=NonNegativeReals))
        setattr(model,f'obj{i}_menos',pyo.Var(domain=NonNegativeReals))
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
       
        return model.obj1_menos + model.obj2_menos
    model.objetivo = pyo.Objective(rule=func_objetivo_1, sense=pyo.minimize)

    #=============================
    # RESTRIÇÕES
    #=============================

    # # OBJ 1 2 3 - Rest 1 2 3
    # def retorno_restr_rule(model,diad):
    #     return sum(model.x[a]*model.retornos_ativos[dia,a] for a in model.ativos for dia in model.dias) + model.obj0_menos - model.obj0_mais >= model.retornos_ibov[diad]
    # model.retorno_restr = pyo.Constraint(model.dias,rule=retorno_restr_rule)

    def piotroski_restr_rule(model):
        return sum(model.x[a]*model.score_p[a] for a in model.ativos) + model.obj1_menos - model.obj1_mais >= 0.9
    model.piotroski_rest = pyo.Constraint(rule=piotroski_restr_rule)

    def mf_restr_rule(model):
        return sum(model.x[a]*model.score_m[a] for a in model.ativos) + model.obj2_menos - model.obj2_mais >= 0.9
    model.mf_restr = pyo.Constraint(rule=mf_restr_rule)

    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    #peso minimo por acao
    def rule_peso_minimo(model, a):
        return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)


    def cardinalidade_min(model):
        return sum(
            model.y[a] for a in model.ativos
            ) >= model.cardinalidade_valor_min
    model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)


    melhor_pesos = {list(model.nome_ativos.data())[a]: pyo.value(model.x[a]) for a in model.ativos}
    pio = {a: sum(pyo.value(model.x[a])*pyo.value(model.score_p[a]) for a in model.ativos)}
    mf = {a: sum(pyo.value(model.x[a])*pyo.value(model.score_m[a]) for a in model.ativos)}
    carteiras_anuais[int(ano_um)+1] = {
        'pesos':  melhor_pesos,
        # 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_um} -> {melhor_pesos}")
    print(f"{ano_um} Piotroski -> {pio}")
    # print(f"Valores obj Menos: {pyo.value(model.obj0_menos), pyo.value(model.obj1_menos), pyo.value(model.obj2_menos)}")
    print(f"Valores obj Menos: {pyo.value(model.obj1_menos), pyo.value(model.obj1_mais)}")
    print(f"{ano_um} Magic Formula -> {mf}")
    # print(f"Valores obj Menos: {pyo.value(model.obj0_menos), pyo.value(model.obj1_menos), pyo.value(model.obj2_menos)}")
    print(f"Valores obj Menos: {pyo.value(model.obj2_menos), pyo.value(model.obj2_mais)}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass




=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2015
Nao consta model
Score Atualizado
Score Atualizado
Retornos atualizados
-----
2015-12-31 TAMANHOS: Retorno 55, Score PIO 55, Score MF 55
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
2015 -> {'ABEV3': 0.0, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BBSE3': 0.0, 'BEEF3': 0.0, 'BRAP4': 0.0, 'BRKM5': 0.02, 'CMIG4': 0.0, 'COGN3': 0.0, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.02, 'CSMG3': 0.0, 'CSNA3': 0.0, 'CVCB3': 0.0, 'CYRE3': 0.0, 'DIRR3': 0.0, 'ECOR3': 0.0, 'EMBJ3': 0.0, 'ENGI11': 0.0, 'EQTL3': 0.0, 'EZTC3': 0.0, 'FLRY3': 0.20000000000000007, 'GGBR4': 0.0, 'GOAU4': 0.0, 'HYPE3': 0.0, 'ISAE4': 0.20000000000000007, 'ITSA4': 0.0, 'JHSF3': 0.0

In [15]:
carteiras_anuais

{2016: {'pesos': {'ABEV3': 0.0,
   'ANIM3': 0.0,
   'AXIA3': 0.0,
   'AZZA3': 0.0,
   'B3SA3': 0.0,
   'BBSE3': 0.0,
   'BEEF3': 0.0,
   'BRAP4': 0.0,
   'BRKM5': 0.02,
   'CMIG4': 0.0,
   'COGN3': 0.0,
   'CPFE3': 0.0,
   'CPLE3': 0.0,
   'CSAN3': 0.02,
   'CSMG3': 0.0,
   'CSNA3': 0.0,
   'CVCB3': 0.0,
   'CYRE3': 0.0,
   'DIRR3': 0.0,
   'ECOR3': 0.0,
   'EMBJ3': 0.0,
   'ENGI11': 0.0,
   'EQTL3': 0.0,
   'EZTC3': 0.0,
   'FLRY3': 0.20000000000000007,
   'GGBR4': 0.0,
   'GOAU4': 0.0,
   'HYPE3': 0.0,
   'ISAE4': 0.20000000000000007,
   'ITSA4': 0.0,
   'JHSF3': 0.0,
   'KLBN11': 0.0,
   'LREN3': 0.02,
   'MGLU3': 0.0,
   'MOTV3': 0.0,
   'MRVE3': 0.20000000000000007,
   'MULT3': 0.0,
   'PETR3': 0.0,
   'PETR4': 0.0,
   'POMO4': 0.0,
   'PRIO3': 0.0,
   'PSSA3': 0.0,
   'RADL3': 0.02,
   'RAPT4': 0.0,
   'RENT3': 0.0,
   'SBSP3': 0.02,
   'SLCE3': 0.0,
   'SUZB3': 0.0,
   'TAEE11': 0.09999999999999967,
   'TOTS3': 0.0,
   'UGPA3': 0.20000000000000007,
   'USIM5': 0.0,
   'VALE3': 0

In [16]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if v >= vb_peso_minimo:
            linhas.append({'ano': an, 'ativo': k, 'peso': round(v, 4)})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [17]:
df_portfolios

,ano,ativo,peso
0,2016,BRKM5,0.0200
1,2016,CSAN3,0.0200
2,2016,FLRY3,0.2000
3,2016,ISAE4,0.2000
4,2016,LREN3,0.0200
...,...,...,...
105,2026,LREN3,0.0200
106,2026,MULT3,0.2000
107,2026,RENT3,0.2000
108,2026,SBSP3,0.1338


In [19]:
df_portfolios.to_csv('carteiras_gp.csv')
